# 08 — Backtest orchestration

Runs the real IC backtest end to end over the last cycles of the slice: build features → train on the trailing window → score → rank → measure the Information Coefficient against the *next* cycle's realized forward returns.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## What is being measured, and what is not

This strategy is a pure ranking system: no positions, no weights, no capital, no orders. A P&L/equity-curve backtest has no meaning here — there is nothing to hold, size, or realize a return on as a portfolio.

Instead this evaluates the *ranking's quality* directly, the standard approach for a cross-sectional ranking system: per cycle, rank the universe, then measure the Spearman rank correlation between each instrument's score and its realized forward return to the next cycle.

The IC and the decile spread are **descriptive statistics about the ranking**, never a simulated trading strategy. No capital is implied anywhere in this computation.

## Running the real orchestration

`run_ic_backtest` is given the slice-wide features and labels that were already built from the real data (an explicitly supported parameter — it otherwise recomputes them internally). Everything else — the per-cycle training, scoring, ranking and IC measurement — runs live here.

Handing it labels for the whole slice is point-in-time safe by construction: `build_training_dataset` excludes any quarter at or after its own target, and drops any row whose label was not yet knowable at that cycle's cutoff, regardless of what the dictionary happens to hold.

In [ ]:
if AVAILABLE:
    import time

    from atlas_quant.backtest.multi_factor_ranking_runner import run_ic_backtest
    from atlas_quant.strategies.multi_factor_ranking_ml.estimator import build_lgbm_ranker_estimator
    from atlas_quant.strategies.multi_factor_ranking_ml.sector_encoding import SectorEncoder

    bundle = rd.slice_bundle()
    config = rd.config()
    cycles = bundle.demo_cycles

    print("evaluating cycles:", [c.quarter_start.isoformat() for c in cycles])
    print("training history available:",
          f"{len(bundle.labeled_quarters)} labeled quarters "
          f"({min(bundle.labeled_quarters)} .. {max(bundle.labeled_quarters)})")
    print()

    started = time.time()
    result = run_ic_backtest(
        config=config,
        universe=bundle.universe,
        fundamentals_by_instrument={},
        prices_by_instrument=bundle.prices_by_instrument,
        sector_encoder=SectorEncoder(),
        cycles=cycles,
        estimator_factory=build_lgbm_ranker_estimator,
        feature_results=bundle.feature_results,
        labeled_by_quarter=bundle.labeled_quarters,
    )
    print(f"completed in {time.time() - started:.1f}s")
else:
    result = None
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Per-cycle results

In [ ]:
if AVAILABLE:
    print(f"{'cycle':<12} {'training':<32} {'ranked':>8} {'scored':>8} "
          f"{'IC':>9} {'decile spread':>14}")
    for cycle_result in result.cycle_results:
        ic = "—" if cycle_result.ic is None else f"{cycle_result.ic:9.4f}"
        spread = "—" if cycle_result.decile_spread is None else f"{cycle_result.decile_spread:14.4f}"
        print(f"{str(cycle_result.cycle.quarter_start):<12} "
              f"{cycle_result.training_state.value:<32} "
              f"{cycle_result.ranked_count:>8,} {cycle_result.scored_for_ic_count:>8,} "
              f"{ic:>9} {spread:>14}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Why the final cycle has no IC

It is ranked, but there is no next cycle to realize a forward return against yet. It still appears in `cycle_results` with `ic=None` — the caller always sees every requested cycle, never a silently shorter list.

In [ ]:
if AVAILABLE:
    last = result.cycle_results[-1]
    print("last cycle           ", last.cycle.quarter_start)
    print("ranked               ", f"{last.ranked_count:,}")
    print("scored for IC        ", last.scored_for_ic_count)
    print("IC                   ", last.ic)
    print()
    print("cycles requested     ", result.cycle_count)
    print("cycles with an IC    ", result.measured_cycle_count)
    print("min_scored_count gate", result.min_scored_count)
    print()
    print("Cycles scored against fewer than min_scored_count candidates are excluded from")
    print("the headline statistics: a rank correlation over a handful of names is")
    print("sampling noise, not signal. They remain visible in cycle_results regardless.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Run identity and audit trail

In [ ]:
if AVAILABLE:
    print("strategy        ", result.strategy_id, result.strategy_version)
    print("config identity ", result.config_identity)
    print("run identity    ", result.run_identity)
    print("warnings        ", result.warnings or "none")
    print()
    for record in result.audit_trail.records:
        print(f"  [{record.stage}] {record.message}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- The real orchestration runs end to end: each evaluated cycle refits the ranker on its own trailing window, scores and ranks ~1,400 real instruments, and measures IC against the following quarter's realized returns.
- The final cycle is ranked but unmeasurable by construction, and reports `ic=None` rather than being dropped or given a placeholder value.
- No equity curve, position, weight or capital figure is produced anywhere in this path, because none exists.

## Limitations

- Only the last few cycles are re-run live here, for runtime. The slice-wide result is in notebook 09.
- IC is measured per cycle over a single quarter's cross-section. A handful of quarterly observations is far too few to establish that the mean IC is distinguishable from zero.
- The cycles evaluated here are recent, and the universe is survivorship-biased. Both inflate apparent quality relative to what was achievable in real time.